In [2]:
## Imports and config
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
import random
import re
import copy

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]
company_names= json.load(open(Path("./json_files/company_names_clean.json")))
features_names ={"Net Income":"netIncome", "Revenue":"totalRevenue","Free Cash Flow":"fcf","Intrinsic Value":"iv","Diluted Shares":"diluted_shares"}


/home/erfan/miniconda3/envs/hf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class QueryGenerator:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=512, temperature=2.0):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p= 0.8,
            top_k=20,
            min_p=0 
            
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        model_output = self.tokenizer.decode(
            output_ids,
            skip_special_tokens=True,
        ).strip("\n")

        return model_output

In [3]:
## Helper functions

def sample_spec(company_names, features_names, min_year=2006, max_year=2025):
    company_names_list = list(company_names.keys())
    feature_names_list = list(features_names.keys())
    companies = random.sample(company_names_list, k=random.randint(1, 2))
    features = random.sample(feature_names_list, k=random.randint(1, 3))

    start_year = random.randint(min_year, max_year)
    end_year = random.randint(start_year, max_year)
    return companies,features,start_year,end_year
    
def sample_spec_intermediate(company_names, features_names, min_year=2006, max_year=2025):
    company_names_list = list(company_names.keys())
    feature_names_list = list(features_names.keys())

    companies = random.sample(company_names_list, k=random.randint(2, 3))
    features = random.sample(feature_names_list, k=random.randint(3, 4))

    company_split_index = len(companies) // 2
    feature_split_index = len(features) // 2

    company_feature_groups = [
        {
            "companies": companies[:company_split_index],
            "features": features[:feature_split_index],
        },
        {
            "companies": companies[company_split_index:],
            "features": features[feature_split_index:],
        },
    ]

    start_year = random.randint(min_year, max_year)
    end_year = random.randint(start_year, max_year)
    return company_feature_groups, start_year, end_year


def generate_user_pormpt(companies:list,features:list,start_year:int,end_year:int):
    return f"""Input specification:
    * Companies: {set(companies)}
    * Metrics: {set(features)}
    * Start year: {str(start_year)}
    * End year: {str(end_year)}
    Your output :
    """


def generate_user_pormpt_intermediate(company_feature_groups:list,start_year:int,end_year:int):
    group_lines = []
    for group_number, group in enumerate(company_feature_groups, start=1):
        group_lines.append(
            f"    * Group {group_number} companies: {set(group['companies'])}\n"
            f"    * Group {group_number} metrics: {set(group['features'])}"
        )

    return f"""Input specification:
    * Use the same year range for every group.
{chr(10).join(group_lines)}
    * Start year: {str(start_year)}
    * End year: {str(end_year)}
    Your output :
    """


def build_expected_output(companies, features, start_year, end_year):
    return {
    "action": "call",
    "function": "get_fundamentals",
    "arguments": {
        "queries": [
            {
                "symbols": [company],
                "metrics": [feature for feature in features],
                "start_year": start_year,
                "end_year": end_year,
            }
            for company in companies
            ]
                },
            }
       

def build_expected_output_intermediate(company_feature_groups, start_year, end_year):
    return {
        "action": "call",
        "function": "get_fundamentals",
        "arguments": {
            "queries": [
                {
                    "symbols": [company for company in group["companies"]],
                    "metrics": [feature for feature in group["features"]],
                    "start_year": start_year,
                    "end_year": end_year,
                }
                for group in company_feature_groups
            ]
        },
    }


def parse_model_output(model_output, expected_count=3):
    import re

    pattern = r"Q\d+:\s*(.*?)(?=\nQ\d+:|$)"
    queries = re.findall(pattern, model_output, flags=re.S)
    queries = [q.strip() for q in queries if q.strip()]

    if len(queries) != expected_count:
        return None

    return queries


def create_data_points(
    model_output,
    expected_output,
    companies,
    features,
    start_year,
    end_year,
    company_names,
    features_names,
    iteration=None,
    ):
    queries = parse_model_output(model_output, expected_count=2)

    if queries is None:
        failed_record = {
            "iteration": iteration,
            "raw_model_output": model_output,
            "expected_output": expected_output,
            "spec": {
                "companies": companies,
                "symbols": [company_names[company] for company in companies],
                "features": features,
                "metrics": [features_names[feature] for feature in features],
                "start_year": start_year,
                "end_year": end_year,
            },
        }

        return [], failed_record



    metadata = {
        "iteration": iteration,
        "companies": companies,
        "symbols": [company_names[company] for company in companies],
        "features": features,
        "metrics": [features_names[feature] for feature in features],
        "start_year": start_year,
        "end_year": end_year,
        "raw_model_output": model_output,
    }

    data_points = [
        {
            "query": query,
            "output_str": json.dumps(expected_output, ensure_ascii=False),
            "output_json": copy.deepcopy(expected_output),
            "metadata": copy.deepcopy(metadata),
        }
        for query in queries
    ]

    return data_points, None


def append_jsonl(path, records):
    with open(path, "a", encoding="utf-8") as fp:
        for record in records:
            fp.write(json.dumps(record, ensure_ascii=False) + "\n")


In [4]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto",
    
)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15512.95it/s]


In [5]:

# prepare the model input
SYSTEM_PROMPT_EASY = f"""You are an assistant of writing three diffrent querries regarding a dataset. This dataset consist of stock market companies list.
Each company has a CSV file with:
- Net income
- Instrinsic Value
- Revenue
- Diluted shares
as metrics, and mostly starts from 2009 to 2025. 

Your resposibilty is to create three queries for function calling. Your assistant will use your quesries and look for the corresponding stocks ticker and it features on specific CSV file.
The user gives you some input specification such as name of comapnies, Metrics (features) and period of time. You must generate the query based on this informattion  
In your query you must used vide range of vocabularies and use different grammers. Put yourself in postion of the user and use natural, common terms and words.
These three qurries should not be  similar to each other from grammer and vocabulary point of view.

You ourput should be like this:
Q1: 
Q2:
Q3:

Rules:
* Preserve every company, metric, and year exactly.
* Do not add or remove any requested information.
* You may vary the sentence structure and the order in which the year range, companies, and metrics appear.
* Do not use pronouns such as “it,” “the former,” or “the latter.”
* Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.
"""


SYSTEM_PROMPT_INTERMEDIATE = """You are an assistant of writing two different queries regarding a dataset. This dataset consists of stock market companies.
Each company has a CSV file with metrics such as:
- Net income
- Intrinsic Value
- Revenue
- Diluted shares
- Free Cash Flow

Your responsibility is to create two natural-language queries for function calling. The user gives grouped input specifications. Each group has its own companies and metrics, and all groups share the same start year and end year.

In every generated query, clearly preserve which companies belong with which metrics. For example, if Group 1 companies have Diluted Shares and Group 2 companies have Net Income, Free Cash Flow, and Revenue, the query must not mix those metrics across the wrong companies.

Use a wide range of vocabulary and different grammar. Put yourself in the position of the user and use natural, common terms and words. The thwo queries should not be similar to each other from grammar and vocabulary point of view.
The start of these two queries should not be the same. 
Eac houtput should contais all informations
Use different:

* openings;
* sentence structures;
* positions of the year range;
* grammatical forms;
* transitions between groups.

Do not generate both queries using the pattern:

“What are the ... for the years between ...?”

Rules:
* Preserve every company, metric, group assignment, start year, and end year exactly.
* Do not add or remove any requested information.
* Make the metric-company relationship explicit for every group.
* You may vary the sentence structure and the order in which the year range, companies, and metrics appear.
* Do not use pronouns such as "it," "the former," or "the latter."
* Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.

Your output should be like this:
Q1:
Q2:
"""

In [9]:
all_data_points = []
failed_generations = []
query_generator= QueryGenerator(model,tokenizer,SYSTEM_PROMPT_EASY, max_new_tokens=512,temperature=0.7)


for i in range(700):
        if (i + 1) % 30 == 0:
            print("iteration:", i+1)
        companies, features, start_year,end_year= sample_spec(company_names, features_names)
        user_prompt= generate_user_pormpt(companies,features,start_year,end_year)
        print(user_prompt,end="\r")
        expected_output = build_expected_output(companies,features,start_year,end_year)
        
        model_output = query_generator.generate(user_prompt)
        
        data_points, failed_record = create_data_points(
        model_output=model_output,
        expected_output=expected_output,
        companies=companies,
        features=features,
        start_year=start_year,
        end_year=end_year,
        company_names=company_names,
        features_names=features_names,
        iteration=i,
    )

        if data_points:
            all_data_points.extend(data_points)
        else:
            failed_generations.append(failed_record)

        if (i + 1) % 100 == 0:
            append_jsonl(f"dataset_checkpoint_{i + 1}.jsonl", all_data_points)
            append_jsonl(f"failed_checkpoint_{i + 1}.jsonl", failed_generations)



append_jsonl(f"dataset_checkpoint.jsonl", all_data_points)
append_jsonl(f"failed_checkpoint.jsonl", failed_generations)

Input specification:
    * Companies: {'United Parcel Service'}
    * Metrics: {'Net Income', 'Intrinsic Value'}
    * Start year: 2014
    * End year: 2024
    Your output :
Input specification:
    * Companies: {'Abbott Laboratories'}
    * Metrics: {'Free Cash Flow', 'Diluted Shares', 'Intrinsic Value'}
    * Start year: 2011
    * End year: 2011
    Your output :
Input specification:
    * Companies: {'Oracle', 'Caterpillar'}
    * Metrics: {'Free Cash Flow', 'Net Income', 'Intrinsic Value'}
    * Start year: 2019
    * End year: 2025
    Your output :
Input specification:
    * Companies: {'American International Group, I'}
    * Metrics: {'Free Cash Flow', 'Diluted Shares'}
    * Start year: 2014
    * End year: 2017
    Your output :
Input specification:
    * Companies: {'Visa'}
    * Metrics: {'Free Cash Flow', 'Diluted Shares', 'Net Income'}
    * Start year: 2020
    * End year: 2025
    Your output :
Input specification:
    * Companies: {'International Business Machines', 

In [8]:
# Intermediate dataset generation example
# This keeps the easy dataset code above unchanged.

all_data_points_intermediate = []
failed_generations_intermediate = []
query_generator_intermediate = QueryGenerator(
    model,
    tokenizer,
    SYSTEM_PROMPT_INTERMEDIATE,
    max_new_tokens=512,
    temperature=0.7,
)

for i in range(211,500):
    if (i + 1) % 30 == 0:
        print("intermediate iteration:", i + 1)

    company_feature_groups, start_year, end_year = sample_spec_intermediate(company_names, features_names)
    user_prompt = generate_user_pormpt_intermediate(company_feature_groups, start_year, end_year)
    print(user_prompt, end="\r")
    expected_output = build_expected_output_intermediate(company_feature_groups, start_year, end_year)

    model_output = query_generator_intermediate.generate(user_prompt)

    companies = [company for group in company_feature_groups for company in group["companies"]]
    features = [feature for group in company_feature_groups for feature in group["features"]]
    data_points, failed_record = create_data_points(
        model_output=model_output,
        expected_output=expected_output,
        companies=companies,
        features=features,
        start_year=start_year,
        end_year=end_year,
        company_names=company_names,
        features_names=features_names,
        iteration=i,
    )

    if data_points:
        for data_point in data_points:
            data_point["metadata"]["company_feature_groups"] = copy.deepcopy(company_feature_groups)
        all_data_points_intermediate.extend(data_points)
    else:
        failed_record["spec"]["company_feature_groups"] = copy.deepcopy(company_feature_groups)
        failed_generations_intermediate.append(failed_record)

    if (i + 1) % 60 == 0:
        append_jsonl(f"dataset_intermediate_checkpoint_{i + 1}.jsonl", all_data_points_intermediate)
        append_jsonl(f"failed_intermediate_checkpoint_{i + 1}.jsonl", failed_generations_intermediate)

append_jsonl("dataset_intermediate_checkpoint.jsonl", all_data_points_intermediate)
append_jsonl("failed_intermediate_checkpoint.jsonl", failed_generations_intermediate)


Input specification:
    * Use the same year range for every group.
    * Group 1 companies: {'Broadcom'}
    * Group 1 metrics: {'Diluted Shares', 'Net Income'}
    * Group 2 companies: {'Apartment Investment and Manage'}
    * Group 2 metrics: {'Revenue', 'Free Cash Flow'}
    * Start year: 2006
    * End year: 2018
    Your output :
Input specification:
    * Use the same year range for every group.
    * Group 1 companies: {'Advanced Micro Devices'}
    * Group 1 metrics: {'Free Cash Flow'}
    * Group 2 companies: {'Elevance Health'}
    * Group 2 metrics: {'Diluted Shares', 'Net Income'}
    * Start year: 2006
    * End year: 2016
    Your output :
Input specification:
    * Use the same year range for every group.
    * Group 1 companies: {"Lowe's Companies"}
    * Group 1 metrics: {'Diluted Shares', 'Free Cash Flow'}
    * Group 2 companies: {'Fidelity National Information S', 'S&P Global'}
    * Group 2 metrics: {'Revenue', 'Net Income'}
    * Start year: 2010
    * End year: 

In [ ]:
from pathlib import Path

input_files = [
    Path("./dataset_intermediate_checkpoint.jsonl"),
    Path("./datasetcheckpoint.jsonl"),
]

output_file = Path("dataset.jsonl")

with output_file.open("w", encoding="utf-8") as out_fp:
    for input_file in input_files:
        with input_file.open("r", encoding="utf-8") as in_fp:
            for line in in_fp:
                if line.strip():
                    out_fp.write(line if line.endswith("\n") else line + "\n")